<a href="https://colab.research.google.com/github/muhammedriswanp/ai-ml-learning-lab/blob/main/week5_genai_langchain_agents_multimodal/day_25_RAG_System_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# =============================================================
# WEEK 30
# DAY 25 — RAG System Architecture
# End-to-end: Ingestion → Chunking → Embedding → Indexing → Retrieval → LLM
# Dataset   : wikipedia -> Artificial intelligence
# LLM       :
# Vector DB : ChromaDB (persisted to disk)
# =============================================================


In [2]:
!pip install langchain langchain-community chromadb wikipedia sentence-transformers langchain-huggingface -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [3]:
!pip install langchain-community wikipedia -q

In [4]:
!pip install langchain_chroma -q

In [5]:
import wikipedia
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import textwrap
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_464/2100033311.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


In [6]:
wikipedia.set_user_agent("my-rag-application/1.0 (contact@example.com)")
loader = WikipediaLoader(
    query="Artificial intelligence",
    lang="en",
    load_max_docs=20,
)
docs = loader.load()
print(f"Successfully loaded {len(docs)} documents!")


Successfully loaded 20 documents!


In [7]:
print(type(docs))
print(type(docs[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [8]:
print(docs[0])

page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these

In [9]:
print(docs[0].metadata.keys())


dict_keys(['title', 'summary', 'source'])


In [10]:
print(docs[0].metadata['source'])

https://en.wikipedia.org/wiki/Artificial_intelligence


In [11]:
print("Title:", docs[0].metadata['title'])
print("Content Preview:", docs[0].page_content[:200])

Title: Artificial intelligence
Content Preview: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and dec


In [12]:
type(docs[1])

langchain_core.documents.base.Document

In [13]:
for item in docs[2]:
    print(item)
    print("*"*100)

('id', None)
****************************************************************************************************
('metadata', {'title': 'A.I. Artificial Intelligence', 'summary': 'A.I. Artificial Intelligence (or simply A.I.) is a 2001 American science fiction drama film  written and directed by Steven Spielberg. The screenplay by Spielberg is based on a screen story by Ian Watson adapting the 1969 short story "Supertoys Last All Summer Long" by Brian Aldiss. Set in a futuristic society, the film stars Haley Joel Osment as David, a childlike android uniquely programmed with the ability to love. Jude Law, Frances O\'Connor, Brendan Gleeson and William Hurt star in supporting roles.\nDevelopment of A.I. originally began after producer and director Stanley Kubrick acquired the rights to Aldiss\'s story in the early 1970s. Kubrick hired a series of writers, including Aldiss, Watson, Bob Shaw, and Sara Maitland, until the mid-1990s. The film languished in development hell for years, partly

### Chunking (Text Splitter)

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(docs)


In [15]:
print(f"   Documents in  : {len(docs)}")


   Documents in  : 20


In [16]:
print(f"   Chunks out    : {len(chunks)}")


   Chunks out    : 221


In [17]:
print(f"   Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")


   Avg chunk size: 329 chars


In [18]:
print(chunks[0].page_content)
print("*"*100)
print(chunks[1].page_content)
print("*"*100)
print(chunks[2].page_content)


Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
****************************************************************************************************
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
****************************************************************************************************
The tr

### Embedding

In [19]:
embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
sample_vector = embed_model.embed_query(docs[0].page_content)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
print(f"   Model      : all-MiniLM-L6-v2")
print(f"   Vector dims: {len(sample_vector)}")
print(f"   First 8 values: {[round(v, 4) for v in sample_vector[:8]]}")

   Model      : all-MiniLM-L6-v2
   Vector dims: 384
   First 8 values: [-0.058, -0.0749, -0.0168, -0.0149, 0.0134, -0.0312, 0.0349, 0.0416]


### Indexing (Build ChromaDB vector store)

In [21]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="db"    #Saved to       : ./chroma_db
)

### Retrieval test (before connecting LLM)

In [22]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

In [23]:
query = "What are the applications of AI?"


In [24]:
retrieved = retriever.invoke(query)

print(f"   Top-k : {len(retrieved)} chunks\n")


   Top-k : 3 chunks



In [25]:
for i, doc in enumerate(retrieved):
    print(f"── Retrieved chunk {i+1} ──")
    print(textwrap.fill(doc.page_content[:300], width=65))
    print()

── Retrieved chunk 1 ──
The traditional goals of AI research include learning, reasoning,
knowledge representation, planning, natural language processing,
and perception, as well as support for robotics. To reach these
goals, AI researchers use techniques including state space search
and mathematical optimization, formal l

── Retrieved chunk 2 ──
weapon systems, arms race dynamics, AI safety and alignment,
technological unemployment, AI-enabled misinformation, how to
treat certain AI systems if they have a moral status (AI welfare
and rights), artificial superintelligence and existential risks.

── Retrieved chunk 3 ──
High-profile applications of AI include advanced web search
engines, chatbots, virtual assistants, autonomous vehicles, and
play and analysis in strategy games (e.g., chess and Go). Since
the 2020s, generative AI has become widely available to generate
images, audio, and videos from text prompts.



### Prompt Template

In [26]:
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful assistant. Use ONLY the context
                below to answer the question. If the answer is not in the
                context, say "I don't have enough information."

                Context:
                {context}

                Question: {question}

                Answer:"""
)


In [27]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256,
    temperature=0.1,
    do_sample=True,
    device=0  # GPU
)

llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [28]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [29]:
questions = [
    "What are the applications of AI?",
    "What is machine learning?",
    "How does natural language processing work?",
    "What are the goals of AI research?",
    "What is generative AI?",
]

In [30]:
for i, question in enumerate(questions):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {question}")
    print('='*60)

    answer = rag_chain.invoke(question)

    if "Answer:" in answer:
      answer = answer.split("Answer:")[-1].strip()

    print(f"\nA: {textwrap.fill(answer[:400], width=65)}")


Q1: What are the applications of AI?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



A: AI has a wide range of applications, including:
- Search engines: Google, Bing, and Yahoo are examples of search
engines that use AI to provide personalized search results.
- Chatbots: Chatbots are computer programs that simulate human
conversation and can be used for customer service, sales, and
support.                  - Virtual assistants: Amazon's Alexa,
Goog

Q2: What is machine learning?

A: Machine learning has several benefits, including:
1. Accuracy: Machine learning algorithms can be trained to
perform tasks with high accuracy, which can lead to better
decision-making and outcomes.                  2. Flexibility:
Machine learning algorithms can be trained on a wide range of
data, making them flexible and adaptable to new situations.
3. Speed: Mac

Q3: How does natural language processing work?

A: There are several potential applications of LLMs in the future,
including:                  1. Personalized language models: LLMs
can be trained on large amounts of user data 

```markdown
╔══════════════════════════════════════════════════════════╗
║         DAY 25 — RAG SYSTEM ARCHITECTURE COMPLETE        ║
╠══════════════════════════════════════════════════════════╣
║  OFFLINE PIPELINE                                        ║
║  ├─ WikipediaLoader        → Document objects            ║
║  ├─ RecursiveCharSplitter  → chunks (500/50)             ║
║  ├─ all-MiniLM-L6-v2      → 384-dim vectors              ║
║  └─ ChromaDB (HNSW)       → persisted index              ║
║                                                          ║
║  ONLINE PIPELINE                                         ║
║  ├─ embed query (same model)                             ║
║  ├─ retriever → top-3 chunks (cosine similarity)         ║
║  ├─ format_docs → fill RAG_PROMPT                        ║
║  ├─ TinyLlama-1.1B → generate answer                     ║
║  └─ StrOutputParser → clean string output                ║
║                                                          ║
║  CHAIN STYLE : LCEL ( | pipe syntax )                    ║
║  VECTOR DB   : ChromaDB                                  ║
║  EMBEDDER    : all-MiniLM-L6-v2                          ║
║  LLM         : TinyLlama-1.1B (Colab GPU)                ║
║  DATASET     : Wikipedia — Artificial Intelligence       ║
║  TEST QUERIES: 5                                         ║
╚══════════════════════════════════════════════════════════╝
```